# Level 4 — Black–Litterman Portfolio Construction

**Audience:** analysts who understand covariance matrices and long-only
Markowitz optimization but want a disciplined way to combine market
equilibrium with explicit views.

**Prerequisites:** Levels 1–3D, labelled pandas objects, and basic matrix
notation.

**Learning goals**

1. reverse-engineer market-implied equilibrium excess returns;
2. encode absolute and relative views with a pick matrix;
3. control view uncertainty and calculate posterior returns/covariance;
4. feed the posterior into the existing long-only Markowitz optimizer;
5. record assumptions and sensitivity rather than treating views as facts.

**Outline:** market prior → views → uncertainty → posterior → allocation
comparison → sensitivity → exercise.

All inputs are synthetic. The notebook requires no credentials, network access,
or private data.

## 1. Setup and unit contract

Every return and covariance input below is annual and expressed in decimal
units. `market_weights`, covariance rows/columns, pick-matrix columns, and
posterior outputs share the same asset labels and order.

The Black–Litterman layer estimates returns and covariance. Portfolio
constraints remain the responsibility of the downstream optimizer.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.portfolio import (
    black_litterman_posterior,
    implied_equilibrium_returns,
    maximum_sharpe_ratio,
    proportional_view_uncertainty,
)

## 2. Define a synthetic market portfolio

The market weights are non-negative and sum to one. The covariance matrix is
symmetric, positive semidefinite, and uses the same annual horizon as all view
returns.

In [ ]:
assets = pd.Index(
    ["Global Equity", "Government Bond", "Gold"],
    name="asset",
)
market_weights = pd.Series(
    [0.50, 0.30, 0.20],
    index=assets,
    name="market_weight",
)
covariance = pd.DataFrame(
    [
        [0.0400, 0.0060, 0.0040],
        [0.0060, 0.0100, 0.0015],
        [0.0040, 0.0015, 0.0225],
    ],
    index=assets,
    columns=assets,
)

pd.DataFrame(
    {
        "market_weight": market_weights,
        "annual_volatility": np.sqrt(np.diag(covariance)),
    }
)

## 3. Reverse-optimize equilibrium excess returns

The market-implied prior is

\[
\pi = \delta\Sigma w_{market},
\]

where `risk_aversion` \(\delta\) is positive. These are equilibrium **excess**
returns under the model contract, not historical forecasts.

In [ ]:
risk_aversion = 2.5
prior_returns = implied_equilibrium_returns(
    market_weights,
    covariance,
    risk_aversion=risk_aversion,
)
prior_returns.to_frame()

## 4. Encode one relative and one absolute view

Each row of \(P\) defines a portfolio:

- `equity_vs_bond`: Global Equity minus Government Bond;
- `gold_absolute`: Gold by itself.

The corresponding \(Q\) values say that the first portfolio has a 4% expected
annual excess return and Gold has a 3% expected annual excess return.

In [ ]:
view_labels = pd.Index(
    ["equity_vs_bond", "gold_absolute"],
    name="view",
)
pick_matrix = pd.DataFrame(
    [
        [1.0, -1.0, 0.0],
        [0.0, 0.0, 1.0],
    ],
    index=view_labels,
    columns=assets,
)
views = pd.Series(
    [0.04, 0.03],
    index=view_labels,
    name="view_return",
)

pick_matrix.assign(view_return=views)

## 5. Inspect proportional view uncertainty

The default He–Litterman heuristic uses

\[
\Omega = \operatorname{diag}\left(
\operatorname{diag}(\tau P\Sigma P^\top)
\right).
\]

Only the diagonal view variances are retained. With this proportional default,
`tau` cancels from posterior returns; it still affects posterior covariance.

In [ ]:
tau = 0.05
default_omega = proportional_view_uncertainty(
    covariance,
    pick_matrix,
    tau=tau,
)
default_omega

## 6. Calculate the Black–Litterman posterior

The posterior adjusts the prior only by the view innovation \(Q-P\pi\), scaled
through covariance and view uncertainty. The implementation uses linear solves
rather than explicit matrix inversion.

In [ ]:
posterior = black_litterman_posterior(
    market_weights,
    covariance,
    pick_matrix,
    views,
    risk_aversion=risk_aversion,
    tau=tau,
)

return_comparison = pd.concat(
    [
        posterior.prior_returns.rename("prior"),
        posterior.posterior_returns.rename("posterior"),
    ],
    axis=1,
)
return_comparison["adjustment"] = (
    return_comparison["posterior"] - return_comparison["prior"]
)
return_comparison

In [ ]:
posterior.posterior_covariance

## 7. Verify an equilibrium-view invariant

If \(Q=P\pi\), the views contain no innovation and posterior expected returns
must equal the prior. This is a useful implementation and data-pipeline check.

In [ ]:
equilibrium_views = pick_matrix @ prior_returns
equilibrium_result = black_litterman_posterior(
    market_weights,
    covariance,
    pick_matrix,
    equilibrium_views,
    risk_aversion=risk_aversion,
    tau=tau,
)

pd.DataFrame(
    {
        "prior": equilibrium_result.prior_returns,
        "posterior": equilibrium_result.posterior_returns,
        "difference": (
            equilibrium_result.posterior_returns
            - equilibrium_result.prior_returns
        ),
    }
)

## 8. Feed the posterior into long-only Markowitz

Black–Litterman does not enforce long-only weights. Here both the prior and
posterior estimates are passed to the same fully invested, long-only
maximum-Sharpe optimizer so the comparison isolates the changed estimates.

In [ ]:
risk_free_rate = 0.0
prior_allocation = maximum_sharpe_ratio(
    risk_free_rate,
    posterior.prior_returns,
    covariance,
)
posterior_allocation = maximum_sharpe_ratio(
    risk_free_rate,
    posterior.posterior_returns,
    posterior.posterior_covariance,
)

allocation_comparison = pd.concat(
    [
        market_weights.rename("market"),
        prior_allocation.rename("prior_markowitz"),
        posterior_allocation.rename("black_litterman"),
    ],
    axis=1,
)
allocation_comparison

## 9. Sensitivity to custom view uncertainty

Smaller diagonal values in \(\Omega\) express tighter uncertainty and move the
posterior closer to the views. Larger values keep it closer to equilibrium.
Both matrices below use the same units as squared annual returns.

In [ ]:
tight_omega = pd.DataFrame(
    np.diag([0.0002, 0.0001]),
    index=view_labels,
    columns=view_labels,
)
loose_omega = pd.DataFrame(
    np.diag([0.0200, 0.0100]),
    index=view_labels,
    columns=view_labels,
)

tight = black_litterman_posterior(
    market_weights,
    covariance,
    pick_matrix,
    views,
    risk_aversion=risk_aversion,
    tau=tau,
    view_uncertainty=tight_omega,
)
loose = black_litterman_posterior(
    market_weights,
    covariance,
    pick_matrix,
    views,
    risk_aversion=risk_aversion,
    tau=tau,
    view_uncertainty=loose_omega,
)

pd.concat(
    [
        prior_returns.rename("prior"),
        loose.posterior_returns.rename("loose_views"),
        tight.posterior_returns.rename("tight_views"),
    ],
    axis=1,
)

## 10. Exercise — reverse the relative view

Change `equity_vs_bond` from +4% to −2%, keep the Gold view unchanged, and
answer:

1. Which posterior return changes most?
2. How does the long-only allocation change?
3. Does Gold change even though its view is unchanged? Explain using covariance.
4. Which input assumptions belong in a decision record?

In [ ]:
exercise_views = views.copy()
exercise_views.loc["equity_vs_bond"] = -0.02

# Calculate a new posterior and allocation here.

### Answer scaffold

In [ ]:
exercise_result = black_litterman_posterior(
    market_weights,
    covariance,
    pick_matrix,
    exercise_views,
    risk_aversion=risk_aversion,
    tau=tau,
)
exercise_allocation = maximum_sharpe_ratio(
    risk_free_rate,
    exercise_result.posterior_returns,
    exercise_result.posterior_covariance,
)

pd.concat(
    [
        posterior.posterior_returns.rename("original_return"),
        exercise_result.posterior_returns.rename("exercise_return"),
        posterior_allocation.rename("original_weight"),
        exercise_allocation.rename("exercise_weight"),
    ],
    axis=1,
)

## Interpretation, pitfalls, and extensions

- Keep covariance, prior, views, uncertainty, and risk-free-rate conventions in
  the same units and horizon.
- `market_weights` are model inputs, not a guarantee that the observed market
  is efficient.
- A view row defines a portfolio. Check its signs and scaling before fitting.
- View uncertainty is variance, not an intuitive percentage confidence unless
  a separately documented mapping is used.
- Smaller \(\Omega\) means stronger influence; it does not make a view more
  accurate.
- Posterior covariance still inherits covariance-estimation risk.
- Optimization constraints and transaction costs are downstream decisions.
- Compare allocations across priors, uncertainty settings, and holdout periods
  before treating the result as robust.

Possible extensions include confidence-to-uncertainty mappings, alternative
priors, factor views, turnover-aware optimization, and chronological validation.
Each requires its own explicit contract and tests.